# Exercises: Orthogonality and Orthonormality

**Chapter:** 03-Advanced-Linear-Algebra | **Section:** 05

Eight graded exercises covering orthogonal projections, Gram-Schmidt, QR decomposition, numerical stability, spectral theorem, and machine learning applications.

| # | Topic | Difficulty |
|---|-------|----------|
| 1 | Projection and Pythagorean decomposition | ★ |
| 2 | Gram-Schmidt and QR factorization | ★ |
| 3 | Orthogonal matrices: rotations and Householder | ★ |
| 4 | Modified Gram-Schmidt stability | ★★ |
| 5 | Least-squares: QR vs normal equations | ★★ |
| 6 | Spectral theorem and Rayleigh quotient | ★★ |
| 7 | Orthogonal weight initialization | ★★★ |
| 8 | Rayleigh quotient iteration | ★★★ |

**Theory:** [notes.md](notes.md) | **Interactive demos:** [theory.ipynb](theory.ipynb)

In [ ]:
import numpy as np
import numpy.linalg as la
from scipy.linalg import qr, polar

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [10, 5]
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def check(name, got, expected=None, cond=None, tol=1e-8):
    ok = cond if cond is not None else np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} -- {name}")
    return ok

print('Setup complete.')


---

## Exercise 1 ★ — Projection and Orthogonal Decomposition

Given $\mathbf{v} = (3, 1, 2)^\top$ and subspace $S = \operatorname{span}\{(1, 1, 0)^\top,\; (0, 1, 1)^\top\}$:

- **(a)** Apply Gram-Schmidt to the spanning vectors to get an ONB $\{\mathbf{q}_1, \mathbf{q}_2\}$.
- **(b)** Compute the projection $\mathbf{v}_S = \operatorname{proj}_S(\mathbf{v})$.
- **(c)** Verify: $\mathbf{v} - \mathbf{v}_S \perp \mathbf{q}_1$ and $\perp \mathbf{q}_2$.
- **(d)** Verify Pythagorean theorem: $\|\mathbf{v}\|^2 = \|\mathbf{v}_S\|^2 + \|\mathbf{v} - \mathbf{v}_S\|^2$.

In [ ]:
# Exercise 1 Solution
v = np.array([3., 1., 2.])
a1 = np.array([1., 1., 0.])
a2 = np.array([0., 1., 1.])

# (a) Gram-Schmidt
q1 = a1 / la.norm(a1)
u2 = a2 - (a2 @ q1) * q1
q2 = u2 / la.norm(u2)

print('(a) ONB for S:')
print(f'  q1 = {q1}')
print(f'  q2 = {q2}')
check('  q1 is unit', cond=abs(la.norm(q1) - 1) < 1e-12)
check('  q2 is unit', cond=abs(la.norm(q2) - 1) < 1e-12)
check('  q1 ⊥ q2', cond=abs(q1 @ q2) < 1e-12)

# (b) Projection
v_S = (v @ q1) * q1 + (v @ q2) * q2
print(f'\n(b) Projection v_S = {v_S}')

# (c) Residual orthogonality
residual = v - v_S
print(f'\n(c) Residual v - v_S = {residual}')
check('  residual ⊥ q1', cond=abs(residual @ q1) < 1e-12)
check('  residual ⊥ q2', cond=abs(residual @ q2) < 1e-12)

# (d) Pythagorean theorem
print(f'\n(d) Pythagorean theorem:')
print(f'  ||v||²      = {la.norm(v)**2:.6f}')
print(f'  ||v_S||²    = {la.norm(v_S)**2:.6f}')
print(f'  ||v-v_S||²  = {la.norm(residual)**2:.6f}')
print(f'  Sum         = {la.norm(v_S)**2 + la.norm(residual)**2:.6f}')
check('  ||v||² = ||v_S||² + ||v-v_S||²',
      la.norm(v)**2, la.norm(v_S)**2 + la.norm(residual)**2)


---

## Exercise 2 ★ — Gram-Schmidt and QR Factorization

Let $A = \begin{pmatrix}1&1&0\\1&0&1\\0&1&1\end{pmatrix}$.

- **(a)** Apply Gram-Schmidt to produce $A = QR$.
- **(b)** Verify $Q^\top Q = I$ and $QR = A$.
- **(c)** Solve least squares $\min_\mathbf{x}\|A\mathbf{x}-\mathbf{b}\|$ for $\mathbf{b}=(1,2,3)^\top$ via $R\mathbf{x}=Q^\top\mathbf{b}$.
- **(d)** Verify against `np.linalg.lstsq`.

In [ ]:
# Exercise 2 Solution
A = np.array([[1.,1.,0.],[1.,0.,1.],[0.,1.,1.]])
b = np.array([1.,2.,3.])

# (a) Gram-Schmidt -> QR
def gs_qr(A):
    m, n = A.shape
    Q = np.zeros_like(A)
    R = np.zeros((n, n))
    for j in range(n):
        v = A[:, j].copy()
        for i in range(j):
            R[i,j] = Q[:,i] @ A[:,j]
            v -= R[i,j] * Q[:,i]
        R[j,j] = la.norm(v)
        Q[:,j] = v / R[j,j]
    return Q, R

Q, R = gs_qr(A)
print('(a) QR via Gram-Schmidt:')
print(f'Q =\n{Q}')
print(f'R =\n{R}')

# (b) Verification
print('\n(b) Verification:')
check('  Q^T Q = I', Q.T @ Q, np.eye(3))
check('  Q R = A', Q @ R, A)

# (c) Least squares via QR
from scipy.linalg import solve_triangular
x_qr = solve_triangular(R, Q.T @ b)
print(f'\n(c) Least-squares solution x = {x_qr}')
print(f'    Residual ||Ax - b|| = {la.norm(A @ x_qr - b):.8f}')

# (d) Compare with lstsq
x_lstsq, _, _, _ = la.lstsq(A, b, rcond=None)
print(f'\n(d) np.linalg.lstsq solution: {x_lstsq}')
check('  QR solution matches lstsq', x_qr, x_lstsq)


---

## Exercise 3 ★ — Orthogonal Matrices: Rotations and Householder

- **(a)** Verify $R_{\pi/4}$ is orthogonal.
- **(b)** Show $R_\theta R_\phi = R_{\theta+\phi}$ by multiplication.
- **(c)** Build Householder $H$ s.t. $H\mathbf{a} = \|\mathbf{a}\|\mathbf{e}_1$ for $\mathbf{a}=(3,4)^\top$.
- **(d)** Explain $\det(H) = -1$ geometrically.

In [ ]:
# Exercise 3 Solution

# (a) Rotation matrix at pi/4
theta = np.pi / 4
R_theta = np.array([[np.cos(theta), -np.sin(theta)],
                    [np.sin(theta),  np.cos(theta)]])
print('(a) R_{pi/4}:')
print(R_theta)
check('  R^T R = I', R_theta.T @ R_theta, np.eye(2))
check('  det(R) = 1', cond=abs(la.det(R_theta) - 1) < 1e-12)

# (b) Composition: R_theta @ R_phi = R_{theta+phi}
phi = np.pi / 6
R_phi = np.array([[np.cos(phi), -np.sin(phi)],
                  [np.sin(phi),  np.cos(phi)]])
R_sum = np.array([[np.cos(theta+phi), -np.sin(theta+phi)],
                  [np.sin(theta+phi),  np.cos(theta+phi)]])
print('\n(b) R_theta @ R_phi = R_{theta+phi}:')
check('  R_theta @ R_phi = R_{theta+phi}', R_theta @ R_phi, R_sum)

# (c) Householder for a=(3,4)
a = np.array([3., 4.])
norm_a = la.norm(a)
# Use sign convention: add sign(a[0])*||a|| to avoid cancellation
v = a.copy()
v[0] += np.sign(a[0]) * norm_a
v = v / la.norm(v)
H = np.eye(2) - 2 * np.outer(v, v)
Ha = H @ a
print(f'\n(c) Householder reflector for a = {a}:')
print(f'  H =\n{H}')
print(f'  H @ a = {Ha}  (should be ±{norm_a:.1f}*e1)')
check('  H is orthogonal', H.T @ H, np.eye(2))
check('  Ha has zero 2nd component', cond=abs(Ha[1]) < 1e-12)
check('  |Ha| = |a|', abs(Ha[0]), norm_a)

# (d) det(H) = -1
print(f'\n(d) det(H) = {la.det(H):.4f}')
print('Geometric reason: H is a REFLECTION (mirrors space across a hyperplane).')
print('Reflections reverse orientation -> det = -1.')
print('A pure rotation (det=+1) cannot map a to -||a||e1 with a single transformation.')


---

## Exercise 4 ★★ — Modified Gram-Schmidt Stability

- **(a)** Implement CGS and MGS.
- **(b)** Test on Hilbert matrix $H_{ij}=1/(i+j-1)$ for $n=10$. Compare $\|Q^\top Q - I\|_\infty$.
- **(c)** Plot orthogonality error vs $n$ for both methods.
- **(d)** Explain why MGS is more stable.

In [ ]:
# Exercise 4 Solution

# (a) Implementations
def cgs(A):
    m, n = A.shape
    Q = np.zeros_like(A, dtype=float)
    R = np.zeros((n,n))
    for j in range(n):
        v = A[:,j].astype(float).copy()
        for i in range(j):
            R[i,j] = Q[:,i] @ A[:,j]
            v -= R[i,j] * Q[:,i]
        R[j,j] = la.norm(v)
        Q[:,j] = v / R[j,j]
    return Q, R

def mgs(A):
    m, n = A.shape
    Q = A.astype(float).copy()
    R = np.zeros((n,n))
    for j in range(n):
        R[j,j] = la.norm(Q[:,j])
        Q[:,j] /= R[j,j]
        for k in range(j+1, n):
            R[j,k] = Q[:,j] @ Q[:,k]
            Q[:,k] -= R[j,k] * Q[:,j]
    return Q, R

def hilbert(n):
    return 1.0/(np.arange(1,n+1)[:,None]+np.arange(1,n+1)[None,:]-1)

# (b) n=10
n = 10
H10 = hilbert(n)
Qc, _ = cgs(H10);  Qm, _ = mgs(H10);  Qh, _ = la.qr(H10)
err_c = np.max(np.abs(Qc.T @ Qc - np.eye(n)))
err_m = np.max(np.abs(Qm.T @ Qm - np.eye(n)))
err_h = np.max(np.abs(Qh.T @ Qh - np.eye(n)))
kappa = la.cond(H10)
print(f'(b) Hilbert matrix n={n}, kappa={kappa:.2e}')
print(f'  CGS  ||Q^T Q - I||_inf = {err_c:.2e}')
print(f'  MGS  ||Q^T Q - I||_inf = {err_m:.2e}')
print(f'  HH   ||Q^T Q - I||_inf = {err_h:.2e}')

# (c) Plot over range of n
ns = range(3, 14)
errs_c, errs_m, errs_h = [], [], []
for nn in ns:
    Hn = hilbert(nn)
    Qc2,_ = cgs(Hn); Qm2,_ = mgs(Hn); Qh2,_ = la.qr(Hn)
    errs_c.append(np.max(np.abs(Qc2.T@Qc2-np.eye(nn))))
    errs_m.append(np.max(np.abs(Qm2.T@Qm2-np.eye(nn))))
    errs_h.append(np.max(np.abs(Qh2.T@Qh2-np.eye(nn))))

if HAS_MPL:
    plt.figure(figsize=(9,5))
    plt.semilogy(list(ns), errs_c, 'r-o', label='CGS', lw=2)
    plt.semilogy(list(ns), errs_m, 'b-s', label='MGS', lw=2)
    plt.semilogy(list(ns), errs_h, 'g-^', label='Householder', lw=2)
    plt.xlabel('n'); plt.ylabel('||Q^T Q - I||_inf')
    plt.title('Orthogonality Loss on Hilbert Matrix')
    plt.legend(); plt.tight_layout(); plt.show()
else:
    print('\n(c) Error comparison (n vs method):')
    print(f'  {"n":>4}  {"CGS":>12}  {"MGS":>12}  {"Householder":>14}')
    for i,nn in enumerate(ns):
        print(f'  {nn:>4}  {errs_c[i]:12.2e}  {errs_m[i]:12.2e}  {errs_h[i]:14.2e}')

print('\n(d) MGS stability explanation:')
print('In CGS, all projections use the original a_j (accumulated rounding error).')
print('In MGS, each step uses the CURRENT v (already partially orthogonalized).')
print('Error bound: CGS = O(eps * kappa^2) vs MGS = O(eps * kappa).')
print('For Hilbert matrix kappa ~ 10^13, difference is ~10^13x!')


---

## Exercise 5 ★★ — Least-Squares Stability: QR vs Normal Equations

Polynomial fitting with a Vandermonde design matrix.

- **(a)** Build $A \in \mathbb{R}^{20 \times 11}$ with $A_{ij} = x_i^{j-1}$.
- **(b)** Solve via (i) normal equations and (ii) QR.
- **(c)** Compute $\kappa(A)$ and $\kappa(A^\top A)$. What is their ratio?
- **(d)** Perturb $\mathbf{y}$ and compare solution sensitivity.

In [ ]:
# Exercise 5 Solution
m, d = 20, 10  # 20 points, degree-10 polynomial
x = np.linspace(0, 1, m)
y_true = np.sin(2*np.pi*x)
y = y_true + 0.05 * np.random.randn(m)

# (a) Design matrix
A = np.vander(x, d+1, increasing=True)
print(f'(a) Design matrix shape: {A.shape}')

# (b) Both solvers
# (i) Normal equations
ATA = A.T @ A
c_ne = la.solve(ATA, A.T @ y)

# (ii) QR
Q_qr, R_qr = la.qr(A)          # full QR, m x m
c_qr = la.solve(R_qr[:d+1, :], Q_qr[:, :d+1].T @ y)

res_ne = la.norm(A @ c_ne - y)
res_qr = la.norm(A @ c_qr - y)
print(f'\n(b) Residuals:')
print(f'  Normal equations: {res_ne:.6e}')
print(f'  QR:               {res_qr:.6e}')

# (c) Condition numbers
kA = la.cond(A)
kATA = la.cond(ATA)
print(f'\n(c) Condition numbers:')
print(f'  kappa(A)   = {kA:.3e}')
print(f'  kappa(A^T A) = {kATA:.3e}')
print(f'  Ratio kappa(A^T A)/kappa(A)^2 = {kATA/kA**2:.4f}  (should be ~1)')

# (d) Perturbation test
n_trials = 50
eps = 1e-4
deltas_ne, deltas_qr = [], []
for _ in range(n_trials):
    dy = eps * np.random.randn(m)
    dc_ne = la.solve(ATA, A.T @ dy)
    dc_qr = la.solve(R_qr[:d+1,:], Q_qr[:,:d+1].T @ dy)
    deltas_ne.append(la.norm(dc_ne) / la.norm(dy))
    deltas_qr.append(la.norm(dc_qr) / la.norm(dy))

print(f'\n(d) Solution sensitivity ||delta_c||/||delta_y|| (averaged):')
print(f'  Normal equations: {np.mean(deltas_ne):.3e}')
print(f'  QR:               {np.mean(deltas_qr):.3e}')
print('QR is more stable when condition number is large.')


---

## Exercise 6 ★★ — Spectral Theorem and Rayleigh Quotient

Let $A = \begin{pmatrix}4&2\\2&1\end{pmatrix}$.

- **(a)** Find eigenvalues and eigenvectors analytically.
- **(b)** Verify eigenvectors are orthogonal; normalize to form $Q$.
- **(c)** Verify $A = Q\Lambda Q^\top$.
- **(d)** Compute $R_A(\mathbf{x})$ for test vectors; verify in $[\lambda_{\min},\lambda_{\max}]$.
- **(e)** Is $A$ positive definite?

In [ ]:
# Exercise 6 Solution
A = np.array([[4., 2.], [2., 1.]])

# (a) Analytical eigenvalues: char poly (4-l)(1-l) - 4 = l^2 - 5l = 0
# => l(l-5) = 0 => lambda = 0, 5
lam1, lam2 = 0.0, 5.0
print(f'(a) Eigenvalues: lambda_1={lam1}, lambda_2={lam2}')
print(f'   (Analytical: solve det(A - lI) = l^2 - 5l = 0)')

# Eigenvectors: A v = l v
# l=0: (A-0I)v=0 -> 4v1+2v2=0 -> v = (1,-2)/sqrt(5)
# l=5: (A-5I)v=0 -> -v1+2v2=0 -> v = (2,1)/sqrt(5)
v1 = np.array([1., -2.]) / np.sqrt(5)
v2 = np.array([2., 1.]) / np.sqrt(5)
print(f'   v1 = {v1}  (eigvec for lambda=0)')
print(f'   v2 = {v2}  (eigvec for lambda=5)')

# Verify
check('  A v1 = 0 v1', A @ v1, 0 * v1)
check('  A v2 = 5 v2', A @ v2, 5 * v2)

# (b) Orthogonality and Q
print(f'\n(b) <v1, v2> = {v1 @ v2:.2e}  (should be ~0)')
check('  v1 ⊥ v2', cond=abs(v1 @ v2) < 1e-12)
Q = np.column_stack([v1, v2])
check('  Q is orthogonal: Q^T Q = I', Q.T @ Q, np.eye(2))

# (c) Spectral decomposition
Lambda = np.diag([lam1, lam2])
A_recon = Q @ Lambda @ Q.T
print(f'\n(c) Q Lambda Q^T =\n{A_recon}')
check('  Q Lambda Q^T = A', A_recon, A)

# (d) Rayleigh quotient
test_vecs = {
    'e1 = (1,0)': np.array([1.,0.]),
    'e2 = (0,1)': np.array([0.,1.]),
    '(1,1)/sqrt2': np.array([1.,1.])/np.sqrt(2),
}
print(f'\n(d) Rayleigh quotients (range [{lam1},{lam2}]):')
for name, xv in test_vecs.items():
    rq = xv @ A @ xv / (xv @ xv)
    in_range = lam1 - 1e-10 <= rq <= lam2 + 1e-10
    print(f'  R_A({name}) = {rq:.4f}   in [{lam1},{lam2}]: {in_range}')

# (e) Positive definiteness
print(f'\n(e) Positive definite? lambda_min = {lam1}')
print('  NO: lambda_min = 0, so A is positive SEMI-definite (singular).')
print(f'  det(A) = {la.det(A):.6f} (= 0 confirms singularity)')
x_test = np.array([1., -2.])
print(f'  x^T A x for x=v1={x_test}: {x_test @ A @ x_test:.6f}  (= 0, confirms PSD not PD)')


---

## Exercise 7 ★★★ — Orthogonal Weight Initialization

Compare gradient stability: Gaussian vs Orthogonal initialization in deep linear networks.

- **(a)** Generate chains $W_L \cdots W_1$ for both inits; compute $\|\text{product}\|_2$.
- **(b)** Plot spectral norm vs depth $L$.
- **(c)** Propagate a gradient vector and compare norms.
- **(d)** Explain using the isometry property.

In [ ]:
# Exercise 7 Solution
n_dim = 32
depths = [1, 2, 4, 8, 16, 32]
n_trials = 20

g_means, o_means = [], []
g_stds, o_stds = [], []

for L in depths:
    gn, on = [], []
    for _ in range(n_trials):
        # Gaussian
        Wg = [np.random.randn(n_dim,n_dim)/np.sqrt(n_dim) for _ in range(L)]
        pg = Wg[0].copy()
        for W in Wg[1:]: pg = pg @ W
        gn.append(la.norm(pg, 2))

        # Orthogonal
        Wo = []
        for _ in range(L):
            Q2, _ = la.qr(np.random.randn(n_dim, n_dim))
            Wo.append(Q2)
        po = Wo[0].copy()
        for W in Wo[1:]: po = po @ W
        on.append(la.norm(po, 2))

    g_means.append(np.mean(gn)); g_stds.append(np.std(gn))
    o_means.append(np.mean(on)); o_stds.append(np.std(on))

print('(a) Spectral norm of product W_L...W_1:')
print(f'{'L':>5}  {'Gauss mean':>12}  {'Gauss std':>12}  {'Orth mean':>12}  {'Orth std':>12}')
for i, L in enumerate(depths):
    print(f'{L:>5}  {g_means[i]:12.4f}  {g_stds[i]:12.4f}  {o_means[i]:12.6f}  {o_stds[i]:12.6f}')

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(9,5))
    ax.errorbar(depths, g_means, yerr=g_stds, label='Gaussian', fmt='-o', color='red', capsize=4)
    ax.errorbar(depths, o_means, yerr=o_stds, label='Orthogonal', fmt='-s', color='blue', capsize=4)
    ax.axhline(1, color='gray', ls=':', label='Ideal (=1)')
    ax.set_yscale('log'); ax.set_xlabel('Depth L')
    ax.set_ylabel('||W_L...W_1||_2 (log scale)')
    ax.set_title('Spectral Norm vs Depth: Gaussian vs Orthogonal Init')
    ax.legend(); plt.tight_layout(); plt.show()

# (c) Gradient propagation
g_vec = np.random.randn(n_dim)
g_vec /= la.norm(g_vec)  # unit gradient

def gradient_norm_ratio(Ws, g):
    """||W_1^T ... W_L^T g|| / ||g||"""
    x = g.copy()
    for W in reversed(Ws):
        x = W.T @ x
    return la.norm(x) / la.norm(g)

L_test = 20
ratios_g, ratios_o = [], []
for _ in range(n_trials):
    Wg2 = [np.random.randn(n_dim,n_dim)/np.sqrt(n_dim) for _ in range(L_test)]
    Wo2 = [la.qr(np.random.randn(n_dim,n_dim))[0] for _ in range(L_test)]
    ratios_g.append(gradient_norm_ratio(Wg2, g_vec))
    ratios_o.append(gradient_norm_ratio(Wo2, g_vec))

print(f'\n(c) Gradient norm ratio ||backprop_g||/||g|| for L={L_test}:')
print(f'  Gaussian:   mean={np.mean(ratios_g):.3e}, std={np.std(ratios_g):.3e}')
print(f'  Orthogonal: mean={np.mean(ratios_o):.6f}, std={np.std(ratios_o):.6f}')

print('\n(d) Explanation:')
print('Product of orthogonal matrices is orthogonal: ||Qx|| = ||x|| for all x.')
print('So ||W_1^T ... W_L^T g|| = ||g|| exactly when each W_k is orthogonal.')
print('Gradients neither vanish (||..g|| -> 0) nor explode (||..g|| -> inf).')


---

## Exercise 8 ★★★ — Rayleigh Quotient Iteration

Rayleigh Quotient Iteration converges cubically to an eigenvalue.

- **(a)** Implement the algorithm.
- **(b)** Test on $A = \begin{pmatrix}3&1&0\\1&2&1\\0&1&1\end{pmatrix}$; track $|\rho_k - \lambda_\star|$.
- **(c)** Verify cubic convergence on a log plot.
- **(d)** Compare speed vs power iteration and inverse iteration.

In [ ]:
# Exercise 8 Solution

A = np.array([[3.,1.,0.],[1.,2.,1.],[0.,1.,1.]])
true_eigs = np.sort(np.linalg.eigvalsh(A))
print(f'True eigenvalues: {true_eigs}')

# (a) Rayleigh Quotient Iteration
def rqi(A, x0, n_iter=12):
    x = x0 / la.norm(x0)
    history = []
    rho = x @ A @ x
    history.append(rho)
    for _ in range(n_iter):
        rho = x @ A @ x
        try:
            y = la.solve(A - rho * np.eye(len(A)), x)
        except la.LinAlgError:
            break  # converged: A - rho*I is singular
        x = y / la.norm(y)
        history.append(x @ A @ x)
    return np.array(history), x

# (b) Run from random start
np.random.seed(7)
x0 = np.random.randn(3)
rho_hist, x_conv = rqi(A, x0)

# Find which eigenvalue was converged to
lam_star = true_eigs[np.argmin(np.abs(true_eigs - rho_hist[-1]))]
errors = np.abs(rho_hist - lam_star)

print(f'\n(b) Converged to eigenvalue: {rho_hist[-1]:.10f}')
print(f'   True eigenvalue:          {lam_star:.10f}')
print(f'\nError per iteration:')
for i, e in enumerate(errors):
    print(f'  iter {i}: {e:.3e}')

# (c) Check cubic convergence
print(f'\n(c) Convergence rate check (should be cubic):')
for i in range(min(6, len(errors)-2)):
    if errors[i] > 1e-14 and errors[i+1] > 1e-14:
        rate = np.log(errors[i+1]+1e-300) / np.log(errors[i]+1e-300)
        print(f'  log(e_{i+1})/log(e_{i}) = {rate:.2f}  (cubic -> 3.0)')

if HAS_MPL:
    plt.figure(figsize=(9,5))
    plt.semilogy(range(len(errors)), errors+1e-16, 'b-o', lw=2, label='RQI error')
    plt.xlabel('Iteration'); plt.ylabel('|rho_k - lambda*|')
    plt.title('Rayleigh Quotient Iteration — Cubic Convergence')
    plt.legend(); plt.tight_layout(); plt.show()

# (d) Compare with power iteration and inverse iteration
def power_iter(A, x0, n_iter=50):
    x = x0/la.norm(x0); hist = []
    for _ in range(n_iter):
        x = A @ x; lam = la.norm(x); x /= lam; hist.append(lam)
    return np.array(hist)

def inv_iter(A, x0, shift, n_iter=20):
    x = x0/la.norm(x0); hist = []
    B = A - shift*np.eye(len(A))
    for _ in range(n_iter):
        y = la.solve(B, x); x = y/la.norm(y)
        rho = x @ A @ x; hist.append(rho)
    return np.array(hist)

pi_hist = power_iter(A, x0)
ii_hist = inv_iter(A, x0, shift=true_eigs[-1]+0.1)

pi_errs = np.abs(pi_hist - true_eigs[-1])
ii_errs = np.abs(ii_hist - lam_star)

# Iterations to reach 1e-8 accuracy
def iters_to_tol(errs, tol=1e-8):
    idx = np.where(errs < tol)[0]
    return idx[0] if len(idx) > 0 else len(errs)

print(f'\n(d) Iterations to reach 1e-8 accuracy:')
print(f'  Power iteration:    {iters_to_tol(pi_errs)} iterations (linear convergence)')
print(f'  Inverse iteration:  {iters_to_tol(ii_errs)} iterations (linear convergence)')
print(f'  RQI:                {iters_to_tol(errors)} iterations (CUBIC convergence!)')
print('RQI is faster because its shift rho_k adapts to the current estimate,')
print('making the solve (A - rho_k I)^{-1} extremely effective near convergence.')


---

## Solutions Summary

| Exercise | Key Result | Mathematical Insight |
|----------|-----------|---------------------|
| 1 | Projection + Pythagorean | $\|\mathbf{v}\|^2 = \|\mathbf{v}_S\|^2 + \|\mathbf{v}-\mathbf{v}_S\|^2$ |
| 2 | QR gives stable least squares | $R\mathbf{x} = Q^\top\mathbf{b}$ avoids $\kappa^2$ squaring |
| 3 | Householder det = -1 | Reflections reverse orientation |
| 4 | MGS vs CGS stability | $O(\epsilon\kappa)$ vs $O(\epsilon\kappa^2)$ |
| 5 | Vandermonde condition | $\kappa(A^\top A) \approx \kappa(A)^2$ |
| 6 | $A=\begin{pmatrix}4&2\\2&1\end{pmatrix}$ is PSD not PD | $\lambda_{\min}=0$ (singular) |
| 7 | Orthogonal init preserves gradient norm | $\|Q_1^\top\cdots Q_L^\top\mathbf{g}\| = \|\mathbf{g}\|$ |
| 8 | RQI cubic convergence | Adaptive shift $\rho_k = R_A(\mathbf{x}^{(k)})$ |

**Back to theory:** [theory.ipynb](theory.ipynb) | **Notes:** [notes.md](notes.md)